Итоговая таблица для вашего проекта
Модель	Тип	Учитывает фичи	Явный фидбек	Рекомендация
LightFM (regression)	Гибридная	✅	✅	⭐ попробуйте первой
FM	Гибридная	✅ (попарно)	✅	⭐ для продвинутого сравнения
Item-KNN (content)	Контентная	✅ (только item)	❌ (ранжирует по схожести)	для baseline по контенту
User-KNN (content)	Контентная	✅ (только user)	❌	для baseline по контенту
ALS/SVD	Коллаборативная	❌	✅	для чистого сравнения

## Выбор метрики Отбор кандидатов
На этом этапе наш алгоритм выдает топ-300 игр из всех возможных.
Главная метрика: Recall@K (Полнота), где K = 300
Что означает: Какую долю игр из тестового сета (в которые юзер реально потом сыграет) мы смогли закинуть в эту корзину из 300 кандидатов?


$$Recall@K = \frac{| \text{RelevantItems} \cap \text{TopK} |}{| \text{RelevantItems} |}$$

| $Recall@K$ | Доля релевантных элементов, попавших в топ-K рекомендаций |
| $K$ | Количество рекомендаций, выдаваемых моделью (размер топ-списка) |
| $\text{RelevantItems}$ | Множество всех релевантных элементов для данного пользователя (например, игры, в которые пользователь реально играл) |

Если идеальная для юзера игра не попала в эти 200 кандидатов, CatBoost на втором этапе её никогда не увидит и не порекомендует. 

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    'data/user_games_target.csv', 
    usecols=['steamid', 'appid', 'target']
)

user_to_idx = {user: idx for idx, user in enumerate(df['steamid'].unique())}
item_to_idx = {item: idx for idx, item in enumerate(df['appid'].unique())}

idx_to_user = {idx: user for user, idx in user_to_idx.items()}
idx_to_item = {idx: item for item, idx in item_to_idx.items()}

df['user_idx'] = df['steamid'].map(user_to_idx)
df['item_idx'] = df['appid'].map(item_to_idx)

Здесь мы используем встроенные методы pandas для стратифицированного разбиения (пользовательский Holdout). Берем $20\%$ взаимодействий каждого пользователя в тестовую выборку и жестко фильтруем утечки (cold start), чтобы в тесте не оказалось игр, которых модель не видела при обучении.

In [14]:
test_df = df.groupby('user_idx', group_keys=False).apply(
    lambda x: x.sample(frac=0.2, random_state=42)
)

train_df = df.drop(test_df.index)

train_items = train_df['item_idx'].unique()
test_df = test_df[test_df['item_idx'].isin(train_items)]

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_92540/1591085123.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_df = df.groupby('user_idx', group_keys=False).apply(


# ALS ДОБАВИТЬ ПОИСАНИЕ АЛГОРИТМА


### Преобразование Target
бинаризуем целевую переменную: если `target <= 1`, мы считаем это отсутствием интереса (0), а всё, что больше — наличием интереса (1). 

Так как мы используем модель неявной рекомендательной системы (Implicit ALS), она обучается только на положительных примерах (взаимодействиях). Нули (дизлайки или те игры, в которые пользователь не играл) модель не должна получать на вход в явном виде, она сама обрабатывает их как отсутствие взаимодействия благодаря структуре разреженной матрицы. Поэтому после преобразования мы оставляем в таблицах только строки с положительным таргетом (единицы).

In [15]:
#train_df['target'] = (train_df['target'] > 1).astype(int)
#test_df['target'] = (test_df['target'] > 1).astype(int)

#train_df = train_df[train_df['target'] == 1].reset_index(drop=True)
#test_df = test_df[test_df['target'] == 1].reset_index(drop=True)

### Создание Sparse-матриц 

Мы используем формат CSR (Compressed Sparse Row) из модуля `scipy.sparse`. Он хранит только ненулевые значения (наши единицы) и их координаты (`user_idx`, `item_idx`). Важно задать матрицам одинаковый жесткий размер `(num_users, num_items)`, чтобы индексы в тренировочной и тестовой выборках совпадали и не выходили за границы при обучении и инференсе.

In [16]:
from scipy.sparse import csr_matrix

num_users = len(user_to_idx)
num_items = len(item_to_idx)

train_sparse = csr_matrix(
    (train_df['target'], (train_df['user_idx'], train_df['item_idx'])),
    shape=(num_users, num_items)
)

test_sparse = csr_matrix(
    (test_df['target'], (test_df['user_idx'], test_df['item_idx'])),
    shape=(num_users, num_items)
)

### Обучение модели ALS
Для обучения мы используем класс `AlternatingLeastSquares` из библиотеки `implicit`. 
основные гиперпараметры модели:
* `factors`: размерность скрытых векторов (embeddings) для пользователей и игр. 
* `regularization`: коэффициент регуляризации ($\lambda$) для штрафа за слишком большие веса
* `iterations`: количество эпох.

In [17]:
import implicit

model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20,
    random_state=42
)

model.fit(train_sparse)

100%|██████████| 20/20 [00:02<00:00,  7.28it/s]


In [18]:
def simple_recall_at_k(model, train, test, K=300):
    recalls = []
    for user_id in range(test.shape[0]):
        test_items = set(test[user_id].indices)
        if not test_items:
            continue
        
        ids, scores = model.recommend(user_id, train[user_id], N=K, filter_already_liked_items=True)
        rec_items = ids
        
        hits = len(set(rec_items) & test_items)
        recalls.append(hits / len(test_items))
    
    return sum(recalls) / len(recalls) if recalls else 0.0
recall_300 = simple_recall_at_k(model, train_sparse, test_sparse, K=300)
print(f"Recall@300: {recall_300:.4f}")

Recall@300: 0.7017


### Этап 6: Тюнинг гиперпараметров с помощью Optuna

На этом этапе мы найдем оптимальные параметры для модели ALS

**Что мы варьируем:**
1. `factors` (32 - 256): размерность скрытых векторов.
2. `regularization` (0.01 - 0.1): сила регуляризации для предотвращения переобучения.
3. `alpha` (1 - 50): вес, который мы придаем нашим положительным взаимодействиям. В Implicit ALS матрица уверенности (Confidence Matrix) строится путем умножения нашей матрицы нулей и единиц на коэффициент `alpha`.

In [19]:
import optuna
def objective(trial):
    factors = trial.suggest_int('factors', 32, 256)
    regularization = trial.suggest_float('regularization', 0.01, 0.1)
    alpha = trial.suggest_int('alpha', 1, 50)
    
    train_conf = (train_sparse * alpha).astype('float32')
    
    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=15,
        random_state=42
    )
    
    model.fit(train_conf, show_progress=False)
    
    recall = simple_recall_at_k(model, train_sparse, test_sparse, K=300)
    
    return recall

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("Лучшие параметры:", study.best_params)
print(f"Лучший Recall@300: {study.best_value:.4f}")

[I 2026-05-15 17:38:36,913] A new study created in memory with name: no-name-2e79c44a-6d56-48b0-8e80-0c0893451ae3
[I 2026-05-15 17:38:43,282] Trial 0 finished with value: 0.6832718766815113 and parameters: {'factors': 99, 'regularization': 0.09052564109813561, 'alpha': 49}. Best is trial 0 with value: 0.6832718766815113.
[I 2026-05-15 17:38:52,288] Trial 1 finished with value: 0.6681612468628155 and parameters: {'factors': 123, 'regularization': 0.03773493723081822, 'alpha': 10}. Best is trial 0 with value: 0.6832718766815113.
[I 2026-05-15 17:39:06,375] Trial 2 finished with value: 0.6125060130423485 and parameters: {'factors': 189, 'regularization': 0.07522671356939338, 'alpha': 23}. Best is trial 0 with value: 0.6832718766815113.
[I 2026-05-15 17:39:34,667] Trial 3 finished with value: 0.573988201443261 and parameters: {'factors': 228, 'regularization': 0.027885535616388957, 'alpha': 5}. Best is trial 0 with value: 0.6832718766815113.
[I 2026-05-15 17:39:39,500] Trial 4 finished wit

Лучшие параметры: {'factors': 32, 'regularization': 0.013817205102312232, 'alpha': 29}
Лучший Recall@300: 0.7290


In [20]:
# 1. Извлекаем лучшие параметры и обучаем финальную модель
best_params = study.best_params
train_conf = (train_sparse * best_params['alpha']).astype('float32')

final_model = implicit.als.AlternatingLeastSquares(
    factors=best_params['factors'],
    regularization=best_params['regularization'],
    iterations=20, # Для финальной модели можно поставить чуть больше эпох, чем в поиске
    random_state=42
)
final_model.fit(train_conf)

# 2. Генерация кандидатов
candidates_list = []
K = 300

# Проходим только по тем юзерам, у которых есть данные в тесте
for user_idx in range(test_sparse.shape[0]):
    if test_sparse[user_idx].nnz == 0:
        continue
        
    ids, scores = final_model.recommend(
        user_idx, 
        train_sparse[user_idx], 
        N=K, 
        filter_already_liked_items=True
    )
    
    real_user_id = idx_to_user[user_idx]
    
    # Собираем данные
    for rank, (i_idx, score) in enumerate(zip(ids, scores)):
        clean_idx = int(i_idx) # Защита от numpy-типов
        
        if clean_idx in idx_to_item:
            candidates_list.append({
                'steamid': real_user_id,
                'appid': idx_to_item[clean_idx],
                'score': float(score),
                'rank': rank + 1
            })

# 3. Сохранение в DataFrame и CSV
candidates_df = pd.DataFrame(candidates_list)
candidates_df.to_csv('data/als_candidates_top300.csv', index=False)

print(f"Кандидаты успешно сохранены! Размер таблицы: {candidates_df.shape}")

100%|██████████| 20/20 [00:02<00:00,  8.87it/s]


Кандидаты успешно сохранены! Размер таблицы: (1430700, 4)


In [21]:
import pandas as pd
import pickle

# 1. Восстанавливаем словари (маппинги) из истории
history = pd.read_parquet('artifacts/history_interactions.parquet')
unique_items = history['appid'].unique()

item2idx = {item: idx for idx, item in enumerate(unique_items)}
idx2item = {idx: item for idx, item in enumerate(unique_items)}

# 2. Загружаем твою обученную модель (которую ты сохранила как als_model.pkl)
with open('artifacts/als_model.pkl', 'rb') as f:
    your_model = pickle.load(f)

# 3. Сохраняем всё вместе в виде словаря!
with open('artifacts/als_model_data.pkl', 'wb') as f:
    pickle.dump({
        'model': your_model,
        'item2idx': item2idx,
        'idx2item': idx2item
    }, f)

print("Модель и маппинги успешно сохранены в als_model_data.pkl!")

Модель и маппинги успешно сохранены в als_model_data.pkl!


In [22]:
print("=== СТАТИСТИКА РЕКОМЕНДАЦИЙ ALS ===")

total_catalog = len(idx_to_item)
recommended_unique = candidates_df['appid'].nunique()
coverage_pct = (recommended_unique / total_catalog) * 100
print(f"Покрытие каталога: {recommended_unique} игр из {total_catalog} ({coverage_pct:.2f}%)")

train_popularity = train_df['item_idx'].map(idx_to_item).value_counts()

candidates_df['train_pop'] = candidates_df['appid'].map(train_popularity).fillna(0)

print(f"Медианная популярность рекомендованной игры: {candidates_df['train_pop'].median():.0f} взаимодействий в Train")
print(f"Максимальная популярность рекомендованной игры: {candidates_df['train_pop'].max():.0f}")

print("\nТоп-10 игр, которые модель рекомендует ЧАЩЕ всего (по appid):")
top_recommended = candidates_df['appid'].value_counts().head(10)
print(top_recommended.to_string())

print("\nРаспределение скоров (Confidence Scores):")
print(candidates_df['score'].describe()[['min', '25%', '50%', '75%', 'max', 'mean']].to_string())

# Очищаем память от временной колонки
candidates_df = candidates_df.drop(columns=['train_pop'])

=== СТАТИСТИКА РЕКОМЕНДАЦИЙ ALS ===
Покрытие каталога: 6154 игр из 24700 (24.91%)
Медианная популярность рекомендованной игры: 212 взаимодействий в Train
Максимальная популярность рекомендованной игры: 3439

Топ-10 игр, которые модель рекомендует ЧАЩЕ всего (по appid):
appid
654310    4110
489520    4106
489630    4047
251570    4043
863550    4024
43160     4010
500       4008
440       4000
304390    3994
231430    3993

Распределение скоров (Confidence Scores):
min     0.047517
25%     0.472400
50%     0.696846
75%     0.877909
max     3.501067
mean    0.674214


In [23]:
# 1. Загружаем оригинальные таргеты (т.к. в test_df они теперь все равны 1)
df_orig = pd.read_csv('data/user_games_target.csv', usecols=['steamid', 'appid', 'target'])

# 2. Восстанавливаем реальные steamid и appid для тестовой выборки
test_real = test_df.copy()
test_real['steamid'] = test_real['user_idx'].map(idx_to_user)
test_real['appid'] = test_real['item_idx'].map(idx_to_item)

# Соединяем, чтобы получить оригинальную колонку 'target' (2, 3, 4, 5)
test_analysis = pd.merge(test_real[['steamid', 'appid']], df_orig, on=['steamid', 'appid'], how='inner')

# 3. Проверяем, нашла ли модель эту игру (есть ли она в кандидатах)
# Делаем set пар (юзер, игра) для очень быстрого поиска
recommended_pairs = set(zip(candidates_df['steamid'], candidates_df['appid']))

# Если пара есть в рекомендованных - значит модель игру не пропустила (Hit)
test_analysis['is_found'] = test_analysis.apply(lambda row: (row['steamid'], row['appid']) in recommended_pairs, axis=1)

# 4. Считаем статистику с группировкой по оригинальному таргету
stats = test_analysis.groupby('target')['is_found'].agg(
    Всего_игр_в_тесте='count',
    Модель_нашла='sum'
)

stats['Модель_пропустила'] = stats['Всего_игр_в_тесте'] - stats['Модель_нашла']
stats['Найдено_%'] = (stats['Модель_нашла'] / stats['Всего_игр_в_тесте'] * 100).round(2)
stats['Пропущено_%'] = (stats['Модель_пропустила'] / stats['Всего_игр_в_тесте'] * 100).round(2)

print("=== СТАТИСТИКА ПРОПУСКОВ ALS В РАЗРЕЗЕ ОЦЕНОК ===")
print("Внимание: Игры с оценкой 1 были отфильтрованы на этапе предобработки (target > 1)\n")
print(stats[['Всего_игр_в_тесте', 'Найдено_%', 'Пропущено_%']])

# Общая статистика
total_found = stats['Модель_нашла'].sum()
total_test = stats['Всего_игр_в_тесте'].sum()
print(f"\nИтого найдено: {total_found} из {total_test} ({total_found/total_test*100:.2f}%)")

=== СТАТИСТИКА ПРОПУСКОВ ALS В РАЗРЕЗЕ ОЦЕНОК ===
Внимание: Игры с оценкой 1 были отфильтрованы на этапе предобработки (target > 1)

        Всего_игр_в_тесте  Найдено_%  Пропущено_%
target                                           
1                   12091      53.25        46.75
2                   19251      52.32        47.68
3                   22935      54.56        45.44
4                   15850      56.59        43.41
5                   14315      45.27        54.73

Итого найдено: 44475 из 84442 (52.67%)


# ITEMKNN НАПИСАТЬ ОПИСАНИЕ

In [ ]:
from scipy.sparse import csr_matrix
from implicit.nearest_neighbours import BM25Recommender

df = pd.read_csv('data/user_games_target.csv', usecols=['steamid', 'appid', 'target'])

user_to_idx = {user: idx for idx, user in enumerate(df['steamid'].unique())}
item_to_idx = {item: idx for idx, item in enumerate(df['appid'].unique())}

df['user_idx'] = df['steamid'].map(user_to_idx)
df['item_idx'] = df['appid'].map(item_to_idx)

test_df = df.groupby('user_idx', group_keys=False).apply(lambda x: x.sample(frac=0.2, random_state=42))
train_df = df.drop(test_df.index)

train_items = train_df['item_idx'].unique()
test_df = test_df[test_df['item_idx'].isin(train_items)]

n_users = len(user_to_idx)
n_items = len(item_to_idx)

train_sparse = csr_matrix((train_df['target'], (train_df['user_idx'], train_df['item_idx'])), shape=(n_users, n_items))
test_sparse = csr_matrix((test_df['target'], (test_df['user_idx'], test_df['item_idx'])), shape=(n_users, n_items))

train_sparse = train_sparse.astype(np.float32)
test_sparse = test_sparse.astype(np.float32)

model = BM25Recommender(K=300)
model.fit(train_sparse)

# Теперь должно работать
recall_300 = simple_recall_at_k(model, train_sparse, test_sparse, K=300)
print(f"Recall@300: {recall_300:.4f}")

/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_62489/3702129704.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_df = df.groupby('user_idx', group_keys=False).apply(lambda x: x.sample(frac=0.2, random_state=42))
/Users/lubimaya/.pyenv/versions/3.10.14/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0009768009185791016 seconds
  warnings.warn(
100%|██████████| 24700/24700 [00:01<00:00, 12779.70it/s]


Recall@300: 0.6355


In [ ]:
from scipy.sparse import csr_matrix
from implicit.nearest_neighbours import BM25Recommender

df = pd.read_csv('data/user_games_target.csv', usecols=['steamid', 'appid', 'target'])

user_to_idx = {user: idx for idx, user in enumerate(df['steamid'].unique())}
item_to_idx = {item: idx for idx, item in enumerate(df['appid'].unique())}
idx_to_user = {idx: user for user, idx in user_to_idx.items()}
idx_to_item = {idx: item for item, idx in item_to_idx.items()}

df['user_idx'] = df['steamid'].map(user_to_idx)
df['item_idx'] = df['appid'].map(item_to_idx)

test_df = df.groupby('user_idx', group_keys=False).apply(lambda x: x.sample(frac=0.2, random_state=42))
train_df = df.drop(test_df.index)

train_items = train_df['item_idx'].unique()
test_df = test_df[test_df['item_idx'].isin(train_items)]

n_users = len(user_to_idx)
n_items = len(item_to_idx)

train_sparse = csr_matrix((train_df['target'], (train_df['user_idx'], train_df['item_idx'])), shape=(n_users, n_items))
test_sparse = csr_matrix((test_df['target'], (test_df['user_idx'], test_df['item_idx'])), shape=(n_users, n_items))

train_sparse = train_sparse.astype(np.float32)
test_sparse = test_sparse.astype(np.float32)

model = BM25Recommender(K=300)
model.fit(train_sparse)

# Генерация кандидатов Топ-300
bm25_candidates = []
train_dict_list = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
test_users = test_df['user_idx'].unique()

for u_idx in test_users:
    scores = model.recommend(u_idx, train_sparse, N=300, filter_already_liked_items=False)
    top_items, top_scores = scores
    
    real_user = idx_to_user[u_idx]
    for i_idx in top_items:
        bm25_candidates.append({
            'steamid': real_user,
            'appid': idx_to_item[i_idx]
        })

bm25_candidates_df = pd.DataFrame(bm25_candidates)

# Статистика пропусков
df_orig = pd.read_csv('data/user_games_target.csv', usecols=['steamid', 'appid', 'target'])

test_real = test_df.copy()
test_real['steamid'] = test_real['user_idx'].map(idx_to_user)
test_real['appid'] = test_real['item_idx'].map(idx_to_item)

test_analysis_bm25 = pd.merge(test_real[['steamid', 'appid']], df_orig, on=['steamid', 'appid'], how='inner')

recommended_pairs_bm25 = set(zip(bm25_candidates_df['steamid'], bm25_candidates_df['appid']))
test_analysis_bm25['is_found'] = test_analysis_bm25.apply(lambda row: (row['steamid'], row['appid']) in recommended_pairs_bm25, axis=1)

stats_bm25 = test_analysis_bm25.groupby('target')['is_found'].agg(
    Всего_игр_в_тесте='count',
    Модель_нашла='sum'
)

stats_bm25['Модель_пропустила'] = stats_bm25['Всего_игр_в_тесте'] - stats_bm25['Модель_нашла']
stats_bm25['Найдено_%'] = (stats_bm25['Модель_нашла'] / stats_bm25['Всего_игр_в_тесте'] * 100).round(2)
stats_bm25['Пропущено_%'] = (stats_bm25['Модель_пропустила'] / stats_bm25['Всего_игр_в_тесте'] * 100).round(2)

print("\n=== СТАТИСТИКА ПРОПУСКОВ BM25 В РАЗРЕЗЕ ОЦЕНОК ===")
print(stats_bm25[['Всего_игр_в_тесте', 'Найдено_%', 'Пропущено_%']])

total_found_bm25 = stats_bm25['Модель_нашла'].sum()
total_test_bm25 = stats_bm25['Всего_игр_в_тесте'].sum()
print(f"\nИтого найдено: {total_found_bm25} из {total_test_bm25} ({total_found_bm25/total_test_bm25*100:.2f}%)")

/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_62489/2587186865.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_df = df.groupby('user_idx', group_keys=False).apply(lambda x: x.sample(frac=0.2, random_state=42))
/Users/lubimaya/.pyenv/versions/3.10.14/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0011572837829589844 seconds
  warnings.warn(
100%|██████████| 24700/24700 [00:02<00:00, 12227.02it/s]



=== СТАТИСТИКА ПРОПУСКОВ BM25 В РАЗРЕЗЕ ОЦЕНОК ===
        Всего_игр_в_тесте  Найдено_%  Пропущено_%
target                                           
1                   12091      32.19        67.81
2                   19251      30.35        69.65
3                   22935      27.26        72.74
4                   15850      25.46        74.54
5                   14315      19.83        80.17

Итого найдено: 22859 из 84442 (27.07%)


# USERKNN.   НАПИСАТЬ ОПИСАНИЕ

In [ ]:
user_model = BM25Recommender(K=300)
user_model.fit(train_sparse.T)


def recommend_userknn(user_id, train_matrix, similarity_matrix, N=300):
    user_sims = similarity_matrix[user_id]
    if user_sims.nnz == 0:
        return []
        
    scores = user_sims.dot(train_matrix).toarray().ravel()
    
    liked_items = train_matrix[user_id].indices
    scores[liked_items] = -np.inf
    
    if len(scores) < N:
        N = len(scores)
        
    top_indices = np.argpartition(scores, -N)[-N:]
    top_indices = top_indices[np.argsort(-scores[top_indices])]
    
    valid_recs = [idx for idx in top_indices if scores[idx] > 0]
    return valid_recs

def simple_recall_at_k_userknn(train, test, similarity_matrix, K=300):
    recalls = []
    test_users = np.unique(test.nonzero()[0])
    
    for user_id in test_users:
        test_items = set(test[user_id].indices)
        if not test_items:
            continue
            
        rec_items = recommend_userknn(user_id, train, similarity_matrix, N=K)
        
        hits = len(set(rec_items) & test_items)
        recalls.append(hits / len(test_items))
    
    return sum(recalls) / len(recalls) if recalls else 0.0

recall_300 = simple_recall_at_k_userknn(train_sparse, test_sparse, user_model.similarity, K=300)
print(f"UserKNN (BM25) Recall@300: {recall_300:.4f}")

/Users/lubimaya/.pyenv/versions/3.10.14/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.001631021499633789 seconds
  warnings.warn(
100%|██████████| 5000/5000 [00:00<00:00, 11480.29it/s]


UserKNN (BM25) Recall@300: 0.7095


In [ ]:
user_model = BM25Recommender(K=300)
user_model.fit(train_sparse.T)

# Генерация кандидатов Топ-300
userknn_candidates = []
train_dict_list = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
test_users = test_df['user_idx'].unique()

for u_idx in test_users:
    user_sims = user_model.similarity[u_idx]
    if user_sims.nnz == 0:
        continue
        
    scores = user_sims.dot(train_sparse).toarray().ravel()
    
    if u_idx in train_dict_list:
        scores[train_dict_list[u_idx]] = -np.inf
    
    top_indices = np.argpartition(scores, -300)[-300:]
    top_indices = top_indices[np.argsort(-scores[top_indices])]
    valid_recs = [idx for idx in top_indices if scores[idx] > 0]
    
    real_user = idx_to_user[u_idx]
    for i_idx in valid_recs:
        userknn_candidates.append({
            'steamid': real_user,
            'appid': idx_to_item[i_idx]
        })

userknn_candidates_df = pd.DataFrame(userknn_candidates)

# Статистика пропусков
test_real = test_df.copy()
test_real['steamid'] = test_real['user_idx'].map(idx_to_user)
test_real['appid'] = test_real['item_idx'].map(idx_to_item)

test_analysis_userknn = pd.merge(test_real[['steamid', 'appid']], df_orig, on=['steamid', 'appid'], how='inner')

recommended_pairs_userknn = set(zip(userknn_candidates_df['steamid'], userknn_candidates_df['appid']))
test_analysis_userknn['is_found'] = test_analysis_userknn.apply(lambda row: (row['steamid'], row['appid']) in recommended_pairs_userknn, axis=1)

stats_userknn = test_analysis_userknn.groupby('target')['is_found'].agg(
    Всего_игр_в_тесте='count',
    Модель_нашла='sum'
)

stats_userknn['Модель_пропустила'] = stats_userknn['Всего_игр_в_тесте'] - stats_userknn['Модель_нашла']
stats_userknn['Найдено_%'] = (stats_userknn['Модель_нашла'] / stats_userknn['Всего_игр_в_тесте'] * 100).round(2)
stats_userknn['Пропущено_%'] = (stats_userknn['Модель_пропустила'] / stats_userknn['Всего_игр_в_тесте'] * 100).round(2)

print("\n=== СТАТИСТИКА ПРОПУСКОВ UserKNN (BM25) В РАЗРЕЗЕ ОЦЕНОК ===")
print(stats_userknn[['Всего_игр_в_тесте', 'Найдено_%', 'Пропущено_%']])

total_found_userknn = stats_userknn['Модель_нашла'].sum()
total_test_userknn = stats_userknn['Всего_игр_в_тесте'].sum()
print(f"\nИтого найдено: {total_found_userknn} из {total_test_userknn} ({total_found_userknn/total_test_userknn*100:.2f}%)")

/Users/lubimaya/.pyenv/versions/3.10.14/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0013439655303955078 seconds
  warnings.warn(
100%|██████████| 5000/5000 [00:00<00:00, 11216.82it/s]



=== СТАТИСТИКА ПРОПУСКОВ UserKNN (BM25) В РАЗРЕЗЕ ОЦЕНОК ===
        Всего_игр_в_тесте  Найдено_%  Пропущено_%
target                                           
1                   12091      50.54        49.46
2                   19251      51.26        48.74
3                   22935      54.93        45.07
4                   15850      60.31        39.69
5                   14315      48.40        51.60

Итого найдено: 45066 из 84442 (53.37%)


# ITEM KNN - CONTENT BASED

Данный алгоритм относится к классу **Content-Based Filtering** с использованием подхода **k-Nearest Neighbors (kNN)** на уровне объектов (Item-Item). В отличие от коллаборативной фильтрации, сходство между играми здесь вычисляется не на основе того, кто в них играл, а исключительно на основе их внутренних характеристик (метаданных).

---

#### 1. Векторизация признаков (Feature Representation)
Пусть $I$ — множество всех игр. Для каждой игры $i \in I$ мы формируем вектор признаков $\vec{x}_i \in \mathbb{R}^d$, где $d$ — общая размерность признакового пространства. Вектор $\vec{x}_i$ является конкатенацией (объединением) нескольких подмножеств признаков:

*   **Бинарные и категориальные признаки** (Платформы, Жанры, Разработчики):
    Преобразуются через Multi-Hot Encoding. Представляют собой разреженный вектор из нулей и единиц $\vec{c}_i \in \{0, 1\}^{d_c}$.
*   **Текстовые признаки** (Описание игры):
    Обрабатываются алгоритмом TF-IDF (Term Frequency-Inverse Document Frequency). Вес слова $t$ в описании игры $i$ вычисляется как:
    $$w_{t, i} = \text{TF}(t, i) \times \log\left(\frac{|I|}{\text{DF}(t)}\right)$$
    Где $\text{TF}$ — частота слова в описании, а $\text{DF}$ — количество игр, в описании которых встречается это слово. Это дает вектор $\vec{t}_i \in \mathbb{R}^{d_t}$.
*   **Числовые признаки** (Количество рекомендаций, Год выпуска):
    Из-за экспоненциального распределения популярности применяется логарифмирование, а затем Min-Max нормализация для приведения к отрезку $[0, 1]$:
    $$v_{norm} = \frac{\log(1 + v) - \min}{\max - \min}$$

В итоге игра $i$ описывается единым вектором: $\vec{x}_i = [\vec{c}_i, \vec{t}_i, v_{recs}, v_{year}]$.

---

#### 2. Матрица сходства (Item-Item Similarity)
Чтобы понять, насколько игра $i$ похожа на игру $j$, используется косинусное расстояние. Оно измеряет косинус угла между двумя векторами в многомерном пространстве:

$$sim(i, j) = \cos(\vec{x}_i, \vec{x}_j) = \frac{\vec{x}_i \cdot \vec{x}_j}{\|\vec{x}_i\|_2 \|\vec{x}_j\|_2} = \frac{\sum_{k=1}^{d} x_{i,k} x_{j,k}}{\sqrt{\sum_{k=1}^{d} x_{i,k}^2} \sqrt{\sum_{k=1}^{d} x_{j,k}^2}}$$

Значение $sim(i, j)$ лежит в диапазоне от 0 до 1 (т.к. у нас нет отрицательных признаков).
**kNN ограничение:** Для экономии памяти и вычислительных ресурсов для каждой игры $i$ мы сохраняем только множество $N_K(i)$ — $K$ самых похожих игр (в нашем коде $K=50$). Для всех остальных игр мы полагаем $sim(i, j) = 0$.

---

#### 3. Генерация предсказаний (Scoring)
Пусть $u$ — целевой пользователь. Его история взаимодействий (Train) задается множеством $H_u$, где каждая игра $i \in H_u$ имеет оценку/таргет $r_{u, i} \in [1, 5]$.

Мы хотим оценить, насколько пользователю $u$ понравится неизвестная ему игра $j \notin H_u$. Ожидаемый скор предсказания $\hat{r}_{u, j}$ вычисляется как взвешенное среднее оценок пользователя на основе сходства игр:

$$\hat{r}_{u, j} = \frac{\sum_{i \in H_u \cap N_K(j)} sim(i, j) \cdot r_{u, i}}{\sum_{i \in H_u \cap N_K(j)} sim(i, j)}$$

*   Числитель: мы смотрим на все игры $i$ из истории пользователя, которые похожи на кандидата $j$. Умножаем сходство на поставленную пользователем оценку.
*   Знаменатель: сумма весов (сходств) для нормализации.

После расчета скора $\hat{r}_{u, j}$ для всех игр-кандидатов, алгоритм сортирует их по убыванию и выдает **Top-N** рекомендаций.

---

#### 4. Оценка качества (Recall@K)
Функция оценки проверяет, насколько хорошо модель предсказывает будущее. 
Пусть $T_u$ — множество игр, в которые пользователь $u$ реально сыграл в будущем (данные из Test), а $R_u$ — множество из $K$ рекомендаций, выданных алгоритмом.

Метрика **Recall (Полнота)** для одного пользователя вычисляется как доля угаданных игр от всех игр в Test:

$$Recall_u@K = \frac{|T_u \cap R_u|}{|T_u|}$$

Итоговая метрика по всей модели — это усредненный Recall по всем пользователям $U_{test}$, у которых тестовое множество не пустое:

$$Recall@K = \frac{1}{|U_{test}|} \sum_{u \in U_{test}} Recall_u@K$$

In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

games = pd.read_csv('data/game_details.csv')

Очистка и парсинг: Преобразуем строки, содержащие списки (например, разработчики, жанры), в реальные списки Python. Бинарные колонки (платформы, бесплатность) переводим в формат 1/0.

In [ ]:
def parse_list_string(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

list_cols = ['developers', 'publishers', 'genres', 'categories']
for col in list_cols:
    games[col] = games[col].fillna('[]').apply(parse_list_string)

bool_cols = ['is_free', 'coming_soon', 'platforms_windows', 'platforms_mac', 'platforms_linux']
for col in bool_cols:
    games[col] = games[col].fillna(False).astype(bool).astype(int)

Категориальные признаки: Применяем MultiLabelBinarizer к колонкам со списками. Он создаст отдельные столбцы из нулей и единиц для каждого уникального жанра, категории, разработчика и издателя.

In [ ]:
mlb = MultiLabelBinarizer()
cat_features_list = []

for col in list_cols:
    encoded = mlb.fit_transform(games[col])
    encoded_df = pd.DataFrame(encoded, columns=[f"{col}_{c}" for c in mlb.classes_])
    cat_features_list.append(encoded_df)

cat_features = pd.concat(cat_features_list, axis=1)

Текстовые признаки: Обрабатываем short_description с помощью TfidfVectorizer. Убираем английские стоп-слова и ограничиваем словарь до 300 самых важных слов (чтобы матрица не получилась слишком тяжелой).

In [ ]:
games['short_description'] = games['short_description'].fillna('')

tfidf = TfidfVectorizer(stop_words='english', max_features=300)
text_matrix = tfidf.fit_transform(games['short_description']).toarray()

text_features = pd.DataFrame(
    text_matrix, 
    columns=[f"tfidf_{w}" for w in tfidf.get_feature_names_out()]
)

Сборка: Объединяем id игры, бинарные фичи платформ, категориальные фичи списков и TF-IDF векторы текстов в одну финальную таблицу признаков item_features, где индексом выступает appid.

Числовые признаки: Обрабатываем recommendations_total. Из-за сильного разброса значений (у популярных игр их миллионы, у инди — единицы) применяем логарифмирование log1p, а затем MinMaxScaler, чтобы привести признак к диапазону от 0 до 1.

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

games['recommendations_total'] = pd.to_numeric(games['recommendations_total'], errors='coerce').fillna(0)

log_recs = np.log1p(games['recommendations_total']).values.reshape(-1, 1)

scaler = MinMaxScaler()
num_features = pd.DataFrame(
    scaler.fit_transform(log_recs), 
    columns=['recommendations_scaled']
)

Даты: Парсим release_date в формат datetime и извлекаем год. Пропуски заполняем медианным значением. Затем также применяем MinMaxScaler, чтобы год стал числовым признаком от 0 до 1.

In [ ]:
games['release_date'] = pd.to_datetime(games['release_date'], errors='coerce')
games['release_year'] = games['release_date'].dt.year
games['release_year'] = games['release_year'].fillna(games['release_year'].median())

date_features = pd.DataFrame(
    scaler.fit_transform(games[['release_year']]), 
    columns=['release_year_scaled']
)

In [ ]:
item_features = pd.concat([
    games[['appid'] + bool_cols],
    cat_features,
    text_features,
    num_features,
    date_features
], axis=1)

item_features = item_features.set_index('appid')

Матрица сходства: Считаем косинусное расстояние между всеми играми на основе полученных признаков. Чтобы не хранить огромную матрицу "все со всеми" в памяти, для каждой игры оставляем только топ-300 самых похожих игр в виде словаря.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(item_features)
appids = item_features.index.tolist()

K = 300
item_sim_dict = {}

for i, current_id in enumerate(appids):
    sim_scores = sim_matrix[i]
    top_indices = np.argsort(sim_scores)[-(K+1):][::-1]
    
    item_sim_dict[current_id] = {
        appids[idx]: sim_scores[idx]
        for idx in top_indices
        if appids[idx] != current_id
    }

Загрузка и сортировка взаимодействий: Загружаем данные пользователей, оставляем только нужные колонки (пользователь, игра, время, таргет). Обязательно сортируем датафрейм по времени последнего запуска rtime_last_played, чтобы не заглядывать в будущее.

In [ ]:
interactions = pd.read_csv('data/user_games_target.csv')
interactions = interactions[['steamid', 'appid', 'rtime_last_played', 'target']]
interactions = interactions.sort_values(['steamid', 'rtime_last_played'])

Train/Test Split: Делим данные хронологически. Для каждого пользователя берем первые 80% его старых игр в Train, а последние 20% сыгранных игр отправляем в Test.

In [ ]:
interactions['user_seq'] = interactions.groupby('steamid').cumcount()
interactions['user_total'] = interactions.groupby('steamid')['appid'].transform('count')

is_test = interactions['user_seq'] >= (interactions['user_total'] * 0.8)

train_df = interactions[~is_test].drop(columns=['user_seq', 'user_total'])
test_df = interactions[is_test].drop(columns=['user_seq', 'user_total'])

Профиль пользователя (User History): Создаем словарь user_history_train, где ключом является steamid, а значением — словарь из игр, в которые он играл в Train, и их оценок (target). Это нужно для генерации рекомендаций и фильтрации.

In [ ]:
user_history_train = train_df.groupby('steamid').apply(
    lambda x: dict(zip(x['appid'], x['target']))
).to_dict()

/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_62489/1015601605.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_history_train = train_df.groupby('steamid').apply(


Функция генерации рекомендаций: Для каждой игры из истории пользователя мы находим похожие игры-кандидаты. Скор кандидата считается как среднее взвешенное: косинусное сходство умножается на оценку пользователя (target от 1 до 5). Игры из Train отфильтровываются, оставшиеся сортируются по убыванию скора.

In [ ]:
def recommend_for_user(user_id, user_history_dict, sim_dict, top_n=300):
    if user_id not in user_history_dict:
        return []
        
    user_history = user_history_dict[user_id]
    candidate_scores = {}
    candidate_sim_sums = {}
    
    for played_item, rating in user_history.items():
        if played_item not in sim_dict:
            continue
            
        for similar_item, similarity in sim_dict[played_item].items():
            if similar_item in user_history:
                continue
                
            if similar_item not in candidate_scores:
                candidate_scores[similar_item] = 0.0
                candidate_sim_sums[similar_item] = 0.0
                
            candidate_scores[similar_item] += similarity * rating
            candidate_sim_sums[similar_item] += similarity
            
    final_scores = {
        item: (candidate_scores[item] / candidate_sim_sums[item])
        for item in candidate_scores if candidate_sim_sums[item] > 0
    }
    
    top_items = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return [item[0] for item in top_items]

Получение предсказаний: Применяем написанную функцию ко всем уникальным пользователям, которые попали в тестовую выборку. Сохраняем результат в словарь predictions, где ключ — steamid, а значение — список из 10 рекомендованных appid.

In [ ]:
test_users = test_df['steamid'].unique()
predictions = {}

for user in test_users:
    predictions[user] = recommend_for_user(
        user_id=user, 
        user_history_dict=user_history_train, 
        sim_dict=item_sim_dict, 
        top_n=300
    )

In [ ]:
# Генерируем топ-300 рекомендаций для теста
predictions_300 = {}
for user in test_users:
    predictions_300[user] = recommend_for_user(
        user_id=user, 
        user_history_dict=user_history_train, 
        sim_dict=item_sim_dict, 
        top_n=300
    )

def recall_at_k(predictions_dict, test_df, K=300):
    recalls = []
    # Группируем тестовые игры по пользователям в виде множеств (set)
    test_dict = test_df.groupby('steamid')['appid'].apply(set).to_dict()
    
    for user_id, test_items in test_dict.items():
        if not test_items:
            continue
        
        # Берем предсказанные K игр
        rec_items = predictions_dict.get(user_id, [])[:K]
        
        # Считаем пересечение
        hits = len(set(rec_items) & test_items)
        recalls.append(hits / len(test_items))
    
    return sum(recalls) / len(recalls) if recalls else 0.0

# Считаем метрику
recall_300 = recall_at_k(predictions_300, test_df, K=300)
print(f"Recall@300: {recall_300:.4f}")

Recall@300: 0.0297


вывод: пользователи играют в разные игры, рекомендация похожих игр - плохая идея

# LIGHTFM НАПИСАТЬ ОПИСАТЬ 

Обработка признаков пользователей: Загружаем данные и оставляем только полезные колонки (страна, статусы аккаунта, дата создания). Заполняем пропуски и переводим все в строковый формат с понятными префиксами (например, country_RU, year_2015). Дату создания (Unix timestamp) переводим в год.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from lightfm.data import Dataset
from lightfm import LightFM
import optuna

games = pd.read_csv('data/game_details.csv')
users = pd.read_csv('data/unique_users.csv')
interactions = pd.read_csv('data/user_games_target.csv')

In [ ]:
def parse_list(x):
    try: return ast.literal_eval(x)
    except: return []

list_cols = ['genres', 'categories']
for col in list_cols:
    games[col] = games[col].fillna('[]').apply(parse_list)

mlb = MultiLabelBinarizer()
cat_features_list = []
for col in list_cols:
    encoded = mlb.fit_transform(games[col])
    encoded_df = pd.DataFrame(encoded, columns=[f"{col}_{c}" for c in mlb.classes_])
    cat_features_list.append(encoded_df)
cat_features = pd.concat(cat_features_list, axis=1)

scaler = MinMaxScaler()
games['recommendations_total'] = pd.to_numeric(games['recommendations_total'], errors='coerce').fillna(0)
log_recs = np.log1p(games['recommendations_total']).values.reshape(-1, 1)
num_feat = pd.DataFrame(scaler.fit_transform(log_recs), columns=['recs_scaled'])

games['release_date'] = pd.to_datetime(games['release_date'], errors='coerce')
games['release_year'] = games['release_date'].dt.year.fillna(games['release_date'].dt.year.median())
year_feat = pd.DataFrame(scaler.fit_transform(games[['release_year']]), columns=['year_scaled'])

bool_cols = ['is_free', 'platforms_windows', 'platforms_mac', 'platforms_linux']
for col in bool_cols:
    games[col] = games[col].fillna(False).astype(bool).astype(int)

item_features = pd.concat([games[['appid'] + bool_cols], cat_features, num_feat, year_feat], axis=1)
item_features = item_features.set_index('appid')

In [ ]:
cols_to_keep = ['steamid', 'loccountrycode', 'personastate', 'communityvisibilitystate', 'timecreated']
users_clean = users[cols_to_keep].copy()

users_clean['loccountrycode'] = users_clean['loccountrycode'].fillna('UNKNOWN').astype(str)
users_clean['feat_country'] = 'country_' + users_clean['loccountrycode']

users_clean['feat_pstate'] = 'pstate_' + users_clean['personastate'].fillna(-1).astype(int).astype(str)
users_clean['feat_vis'] = 'vis_' + users_clean['communityvisibilitystate'].fillna(-1).astype(int).astype(str)

users_clean['timecreated'] = pd.to_numeric(users_clean['timecreated'], errors='coerce')
users_clean['created_year'] = pd.to_datetime(users_clean['timecreated'], unit='s', errors='coerce').dt.year
users_clean['created_year'] = users_clean['created_year'].fillna(users_clean['created_year'].median())
users_clean['feat_year'] = 'year_' + users_clean['created_year'].astype(int).astype(str)

user_features_df = users_clean[['steamid', 'feat_country', 'feat_pstate', 'feat_vis', 'feat_year']]
user_features_df = user_features_df.set_index('steamid')

In [ ]:
interactions_clean = interactions[['steamid', 'appid', 'rtime_last_played', 'target']].copy()
interactions_clean = interactions_clean.sort_values(['steamid', 'rtime_last_played'])

interactions_clean['user_seq'] = interactions_clean.groupby('steamid').cumcount()
interactions_clean['user_total'] = interactions_clean.groupby('steamid')['appid'].transform('count')

is_test = interactions_clean['user_seq'] >= (interactions_clean['user_total'] * 0.8)

train_df = interactions_clean[~is_test].drop(columns=['user_seq', 'user_total'])
test_df = interactions_clean[is_test].drop(columns=['user_seq', 'user_total'])

# Словари для быстрой фильтрации на этапе теста
train_dict = train_df.groupby('steamid')['appid'].apply(set).to_dict()
test_dict = test_df.groupby('steamid')['appid'].apply(set).to_dict()

In [ ]:
unique_users = train_df['steamid'].unique()
unique_items = train_df['appid'].unique()

dataset = Dataset()
dataset.fit(
    users=unique_users, 
    items=unique_items
)
interactions_gen = ((row['steamid'], row['appid'], 1.0) for _, row in train_df.iterrows())
train_interactions, train_weights = dataset.build_interactions(interactions_gen)

user_id_map, _, item_id_map, _ = dataset.mapping()
inv_item_id_map = {v: k for k, v in item_id_map.items()}
all_item_indices = np.array(list(item_id_map.values()))

In [ ]:
"""sample_test_users = list(test_dict.keys())
def objective(trial):
    no_components = trial.suggest_categorical('no_components', [64, 128, 256])
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1, log=True)
    
    item_alpha = trial.suggest_float('item_alpha', 1e-8, 1e-4, log=True)
    user_alpha = trial.suggest_float('user_alpha', 1e-8, 1e-4, log=True)
    
    model = LightFM(loss='warp', no_components=no_components, 
                    learning_rate=learning_rate, item_alpha=item_alpha, 
                    user_alpha=user_alpha, random_state=42)
    
    model.fit(interactions=train_interactions, sample_weight=train_weights, 
              epochs=10, num_threads=4)
    
    recalls = []
    for steamid in sample_test_users:
        if steamid not in user_id_map: continue
        
        scores = model.predict(user_id_map[steamid], all_item_indices, num_threads=4)
        
        if steamid in train_dict:
            train_indices = [item_id_map[i] for i in train_dict[steamid] if i in item_id_map]
            scores[train_indices] = -np.inf
            
        top_300 = np.argpartition(scores, -300)[-300:]
        rec_items = set([inv_item_id_map[idx] for idx in top_300])
        hits = len(rec_items & test_dict[steamid])
        recalls.append(hits / len(test_dict[steamid]))
        
    return sum(recalls) / len(recalls) if recalls else 0

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
print(f"Лучшие параметры: {study.best_params}")"""
sample_test_users = list(test_dict.keys())

def objective(trial):
    # 1. Увеличенный диапазон компонентов
    no_components = trial.suggest_categorical('no_components', [128, 256, 512])
    
    # 2. Более широкий диапазон learning_rate
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.05, log=True)
    
    # 3. Раздельная регуляризация
    item_alpha = trial.suggest_float('item_alpha', 1e-7, 1e-3, log=True)
    user_alpha = trial.suggest_float('user_alpha', 1e-7, 1e-3, log=True)
    
    # 4. Количество эпох
    epochs = trial.suggest_int('epochs', 20, 50)
    
    # 5. ИСПРАВЛЕНО: только loss'ы, совместимые с sample_weights
    loss = trial.suggest_categorical('loss', ['warp', 'bpr'])  # убран 'warp-kos'
    
    # 6. max_sampled (только для warp)
    max_sampled = trial.suggest_int('max_sampled', 5, 15)
    
    model = LightFM(
        loss=loss,
        no_components=no_components,
        learning_rate=learning_rate,
        item_alpha=item_alpha,
        user_alpha=user_alpha,
        max_sampled=max_sampled,
        random_state=42
    )
    
    # ИСПРАВЛЕНО: передаем sample_weight только если не None
    if train_weights is not None:
        model.fit(
            interactions=train_interactions, 
            sample_weight=train_weights, 
            epochs=epochs,
            num_threads=4
        )
    else:
        model.fit(
            interactions=train_interactions, 
            epochs=epochs,
            num_threads=4
        )
    
    recalls = []
    for steamid in sample_test_users:
        if steamid not in user_id_map: continue
        
        scores = model.predict(user_id_map[steamid], all_item_indices, num_threads=4)
        
        if steamid in train_dict:
            train_indices = [item_id_map[i] for i in train_dict[steamid] if i in item_id_map]
            scores[train_indices] = -np.inf
            
        top_300 = np.argpartition(scores, -300)[-300:]
        rec_items = set([inv_item_id_map[idx] for idx in top_300])
        hits = len(rec_items & test_dict[steamid])
        recalls.append(hits / len(test_dict[steamid]))
        
    return sum(recalls) / len(recalls) if recalls else 0

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=25)
print(f"Лучшие параметры: {study.best_params}")

[I 2026-05-15 17:00:27,042] A new study created in memory with name: no-name-7b772784-e0d2-44b6-a0f5-05a8ffe39dd2
[I 2026-05-15 17:01:24,283] Trial 0 finished with value: 0.23878955641392624 and parameters: {'no_components': 256, 'learning_rate': 0.0153673417558194, 'item_alpha': 3.878014289790159e-06, 'user_alpha': 1.9748674014345186e-05, 'epochs': 48, 'loss': 'bpr', 'max_sampled': 7}. Best is trial 0 with value: 0.23878955641392624.
[I 2026-05-15 17:02:11,595] Trial 1 finished with value: 0.17407268631800757 and parameters: {'no_components': 512, 'learning_rate': 0.007302497718920594, 'item_alpha': 4.023673070733259e-07, 'user_alpha': 0.0008524631397018396, 'epochs': 27, 'loss': 'warp', 'max_sampled': 6}. Best is trial 0 with value: 0.23878955641392624.
[I 2026-05-15 17:02:27,157] Trial 2 finished with value: 0.17112508292308312 and parameters: {'no_components': 128, 'learning_rate': 0.005630112857796318, 'item_alpha': 9.170697816246026e-05, 'user_alpha': 0.0004480742194520681, 'epoc

Лучшие параметры: {'no_components': 256, 'learning_rate': 0.024729017055949886, 'item_alpha': 0.0009331014234027413, 'user_alpha': 1.0496013775403137e-05, 'epochs': 50, 'loss': 'bpr', 'max_sampled': 9}


In [ ]:
best_p = study.best_params
final_model = LightFM(
    loss='warp',
    no_components=best_p['no_components'],
    learning_rate=best_p['learning_rate'],
    item_alpha=best_p['item_alpha'],
    user_alpha=best_p['user_alpha'],
    random_state=42
)

final_model.fit(
    interactions=train_interactions,
    sample_weight=train_weights,
    epochs=30,
    num_threads=4
)

recalls_final = []

for steamid, test_items in test_dict.items():
    if steamid not in user_id_map: 
        continue
        
    u_idx = user_id_map[steamid]
    
    scores = final_model.predict(u_idx, all_item_indices, num_threads=4)
    if steamid in train_dict:
        train_indices = [item_id_map[i] for i in train_dict[steamid] if i in item_id_map]
        scores[train_indices] = -np.inf

    top_300_idx = np.argpartition(scores, -300)[-300:]
    rec_items = set([inv_item_id_map[idx] for idx in top_300_idx])
    
    # Метрика
    hits = len(rec_items & test_items)
    recalls_final.append(hits / len(test_items))

final_recall = sum(recalls_final) / len(recalls_final) if recalls_final else 0.0
print(f"PURE CF LightFM Recall@300: {final_recall:.4f}")

PURE CF LightFM Recall@300: 0.2115


In [ ]:
best_p = study.best_params
final_model = LightFM(
    loss='warp',
    no_components=best_p['no_components'],
    learning_rate=best_p['learning_rate'],
    item_alpha=best_p['item_alpha'],
    user_alpha=best_p['user_alpha'],
    random_state=42
)

final_model.fit(
    interactions=train_interactions,
    sample_weight=train_weights,
    epochs=30,
    num_threads=4
)

# Сбор метрик по target (количеству игр в тесте)
target_stats = {1: {'total': 0, 'found': 0}, 2: {'total': 0, 'found': 0}, 
                3: {'total': 0, 'found': 0}, 4: {'total': 0, 'found': 0}, 
                5: {'total': 0, 'found': 0}}

for steamid, test_items in test_dict.items():
    if steamid not in user_id_map: 
        continue
        
    u_idx = user_id_map[steamid]
    test_count = len(test_items)
    
    if test_count not in target_stats:
        continue
    
    scores = final_model.predict(u_idx, all_item_indices, num_threads=4)
    if steamid in train_dict:
        train_indices = [item_id_map[i] for i in train_dict[steamid] if i in item_id_map]
        scores[train_indices] = -np.inf

    top_300_idx = np.argpartition(scores, -300)[-300:]
    rec_items = set([inv_item_id_map[idx] for idx in top_300_idx])
    
    hits = len(rec_items & test_items)
    target_stats[test_count]['total'] += test_count
    target_stats[test_count]['found'] += hits

# Вывод таблицы
print(f"{'target':<6} {'Всего_игр_в_тесте':<18} {'Найдено_%':<12} {'Пропущено_%':<12}")
print("-" * 55)

total_all = 0
found_all = 0

for target in sorted(target_stats.keys()):
    total = target_stats[target]['total']
    found = target_stats[target]['found']
    found_pct = (found / total * 100) if total > 0 else 0
    missed_pct = 100 - found_pct
    
    total_all += total
    found_all += found
    
    print(f"{target:<6} {total:<18} {found_pct:<12.2f} {missed_pct:<12.2f}")

print("-" * 55)
print(f"Итого найдено: {found_all} из {total_all} ({found_all/total_all*100:.2f}%)")

print(f"\nPURE CF LightFM Recall@300: {final_recall:.4f}")

target Всего_игр_в_тесте  Найдено_%    Пропущено_% 
-------------------------------------------------------
1      459                40.74        59.26       
2      722                34.21        65.79       
3      861                32.29        67.71       
4      1036               29.05        70.95       
5      1335               26.07        73.93       
-------------------------------------------------------
Итого найдено: 1361 из 4413 (30.84%)

PURE CF LightFM Recall@300: 0.2115


# FM написать описание

Подготовка данных для PyTorch: Создаем Dataset и DataLoader с негативным сэмплированием. На каждую игру, в которую играл пользователь, мы случайно выбираем одну игру, в которую он не играл.


In [ ]:
import torch
import random
from torch.utils.data import Dataset, DataLoader

class FMDataset(Dataset):
    def __init__(self, train_df, user2idx, item2idx, num_items):
        self.data = []
        user_items = train_df.groupby('steamid')['appid'].apply(lambda x: set(x.map(item2idx).dropna().astype(int))).to_dict()
        self.user_pos_items = {user2idx[u]: items for u, items in user_items.items() if u in user2idx}
        
        for u_idx, items in self.user_pos_items.items():
            for i_idx in items:
                self.data.append((u_idx, i_idx))
        self.num_items = num_items

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        u_idx, pos_idx = self.data[idx]
        neg_idx = random.randrange(self.num_items)
        while neg_idx in self.user_pos_items[u_idx]:
            neg_idx = random.randrange(self.num_items)
        return u_idx, pos_idx, neg_idx

fm_dataset = FMDataset(train_df, user2idx, item2idx, num_items=len(item2idx))
train_loader = DataLoader(fm_dataset, batch_size=2048, shuffle=True)

Архитектура Factorization Machine: Модель вычисляет линейную сумму признаков и их попарные взаимодействия. Для ускорения взаимодействий используется математический трюк: 1/2 * (Сумма_векторов^2 - Сумма_квадратов_векторов).

Обучение модели: Используем оптимизатор Adam и функцию потерь BPR (Bayesian Personalized Ranking), 

In [ ]:
import torch.nn.functional as F
import torch.optim as optim

device = torch.device('mps')
user_feat_tensor = user_feat_tensor.to(device)
item_feat_tensor = item_feat_tensor.to(device)

fm_model = PyTorchFM(
    num_users=len(user2idx), num_items=len(item2idx), 
    u_feat_dim=user_feat_tensor.shape[1], i_feat_dim=item_feat_tensor.shape[1], dim=64
).to(device)

optimizer = optim.Adam(fm_model.parameters(), lr=0.005, weight_decay=1e-5)

fm_model.train()
for epoch in range(7):
    for u, pos, neg in train_loader:
        u, pos, neg = u.to(device), pos.to(device), neg.to(device)
        
        optimizer.zero_grad()
        pos_scores, neg_scores = fm_model(u, pos, neg, user_feat_tensor[u], item_feat_tensor[pos], item_feat_tensor[neg])
        loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
        
        loss.backward()
        optimizer.step()

Быстрый инференс и расчет метрики: Раскрываем формулу FM для матричного умножения всех пользователей на все игры сразу. Полученные предсказания прогоняем через функцию расчета Recall@300.


In [ ]:
import numpy as np

fm_model.eval()
with torch.no_grad():
    all_u = torch.arange(len(user2idx)).to(device)
    all_i = torch.arange(len(item2idx)).to(device)
    
    U_emb = fm_model.u_emb(all_u) + fm_model.uf_proj(user_feat_tensor)
    I_emb = fm_model.i_emb(all_i) + fm_model.if_proj(item_feat_tensor)
    interactions = torch.matmul(U_emb, I_emb.T)
    
    U_lin = fm_model.u_lin(all_u).squeeze() + fm_model.uf_lin(user_feat_tensor).squeeze()
    I_lin = fm_model.i_lin(all_i).squeeze() + fm_model.if_lin(item_feat_tensor).squeeze()
    
    U_int = torch.sum(fm_model.u_emb(all_u) * fm_model.uf_proj(user_feat_tensor), dim=1)
    I_int = torch.sum(fm_model.i_emb(all_i) * fm_model.if_proj(item_feat_tensor), dim=1)
    
    scores = interactions + U_lin.unsqueeze(1) + I_lin.unsqueeze(0) + U_int.unsqueeze(1) + I_int.unsqueeze(0)

scores_np = scores.cpu().numpy()

# Расчет Recall@300
train_dict = train_df.groupby('steamid')['appid'].apply(set).to_dict()
test_dict = test_df.groupby('steamid')['appid'].apply(set).to_dict()
inv_item2idx = {v: k for k, v in item2idx.items()}

recalls = []
for u_id, test_items in test_dict.items():
    if u_id not in user2idx: continue
    
    u_idx = user2idx[u_id]
    u_scores = scores_np[u_idx].copy()
    
    if u_id in train_dict:
        train_indices = [item2idx[i] for i in train_dict[u_id] if i in item2idx]
        u_scores[train_indices] = -np.inf
        
    top_300_idx = np.argpartition(u_scores, -300)[-300:]
    rec_items = set([inv_item2idx[idx] for idx in top_300_idx])
    
    recalls.append(len(rec_items & test_items) / len(test_items))

print(f"PyTorch FM Recall@300: {np.mean(recalls):.4f}")

PyTorch FM Recall@300: 0.1807


In [ ]:
from lightfm import LightFM

# 1. Обучаем LightFM (модель любит формат COO для обучения)
print("Обучение LightFM...")
model_lfm = LightFM(loss='warp', no_components=64, random_state=42)
model_lfm.fit(train_sparse.tocoo(), epochs=20, num_threads=4)

# 2. Генерируем кандидатов Топ-300
print("Генерация кандидатов...")
lfm_candidates = []
all_items = np.arange(num_items)

# Словарь для быстрого зануления Train-игр
train_dict_list = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
test_users = test_df['user_idx'].unique()

for u_idx in test_users:
    # LightFM предсказывает скоры сразу для всех игр
    scores = model_lfm.predict(np.full(len(all_items), u_idx, dtype=np.int32), all_items.astype(np.int32), num_threads=4)
    
    # Зануляем Train
    if u_idx in train_dict_list:
        scores[train_dict_list[u_idx]] = -np.inf
        
    # Берем Топ-300
    top_300_idx = np.argpartition(scores, -300)[-300:]
    
    real_user = idx_to_user[u_idx]
    for i_idx in top_300_idx:
        lfm_candidates.append({
            'steamid': real_user,
            'appid': idx_to_item[i_idx]
        })

lfm_candidates_df = pd.DataFrame(lfm_candidates)

# 3. Анализ качества по оценкам (как у ALS)
df_orig = pd.read_csv('data/user_games_target.csv', usecols=['steamid', 'appid', 'target'])

test_real = test_df.copy()
test_real['steamid'] = test_real['user_idx'].map(idx_to_user)
test_real['appid'] = test_real['item_idx'].map(idx_to_item)

test_analysis_lfm = pd.merge(test_real[['steamid', 'appid']], df_orig, on=['steamid', 'appid'], how='inner')

# Быстрый поиск
recommended_pairs_lfm = set(zip(lfm_candidates_df['steamid'], lfm_candidates_df['appid']))
test_analysis_lfm['is_found'] = test_analysis_lfm.apply(lambda row: (row['steamid'], row['appid']) in recommended_pairs_lfm, axis=1)

# Статистика
stats_lfm = test_analysis_lfm.groupby('target')['is_found'].agg(
    Всего_игр_в_тесте='count',
    Модель_нашла='sum'
)

stats_lfm['Модель_пропустила'] = stats_lfm['Всего_игр_в_тесте'] - stats_lfm['Модель_нашла']
stats_lfm['Найдено_%'] = (stats_lfm['Модель_нашла'] / stats_lfm['Всего_игр_в_тесте'] * 100).round(2)
stats_lfm['Пропущено_%'] = (stats_lfm['Модель_пропустила'] / stats_lfm['Всего_игр_в_тесте'] * 100).round(2)

print("\n=== СТАТИСТИКА ПРОПУСКОВ LightFM (WARP) В РАЗРЕЗЕ ОЦЕНОК ===")
print(stats_lfm[['Всего_игр_в_тесте', 'Найдено_%', 'Пропущено_%']])

total_found_lfm = stats_lfm['Модель_нашла'].sum()
total_test_lfm = stats_lfm['Всего_игр_в_тесте'].sum()
print(f"\nИтого найдено: {total_found_lfm} из {total_test_lfm} ({total_found_lfm/total_test_lfm*100:.2f}%)")

Обучение LightFM...
Генерация кандидатов...

=== СТАТИСТИКА ПРОПУСКОВ LightFM (WARP) В РАЗРЕЗЕ ОЦЕНОК ===
        Всего_игр_в_тесте  Найдено_%  Пропущено_%
target                                           
2                   19251      54.55        45.45
3                   22935      56.39        43.61
4                   15850      58.81        41.19
5                   14315      46.67        53.33

Итого найдено: 39435 из 72351 (54.51%)
